In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
# ============================================================
# CELL 1 — INSTALLATIONS, IMPORTS, CONSTANTS
# ============================================================

# ---- Part A: Installations ---------------------------------
# transformers pinned first — must be 4.44.0 for Phi-3.5-mini

!pip install -q "transformers==4.44.0"
!pip install -q pymupdf
!pip install -q pdfplumber
!pip install -q sentence-transformers
!pip install -q rank-bm25
!pip install -q qdrant-client
!pip install -q neo4j
!pip install -q accelerate

import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

# ---- Part B: Imports ---------------------------------------

import os
import re
import json
import uuid
import shutil
import glob
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from tqdm import tqdm

import fitz
import pdfplumber
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, SparseVectorParams,
    PointStruct, SparseVector, NamedVector,
    NamedSparseVector, Filter,
    FieldCondition, MatchValue
)

from neo4j import GraphDatabase

import torch
import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

# ---- Part C: Version + GPU Check ---------------------------

print(f"Transformers version : {transformers.__version__}")
assert transformers.__version__ == "4.44.0", \
    f"WRONG VERSION: {transformers.__version__} — restart kernel and rerun"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
if device == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU — enable GPU in Kaggle settings")

# ---- Part D: Project folder --------------------------------

PROJECT_DIR = "/kaggle/working/rag_project"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/qdrant_storage", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/images", exist_ok=True)
print(f"\nProject folder : {PROJECT_DIR}")

# ---- Part E: PDF detection ---------------------------------

pdf_files = glob.glob("/kaggle/input/**/*.pdf", recursive=True)

if pdf_files:
    src_path = pdf_files[0]
    uploaded_filename = os.path.basename(src_path)
    PDF_PATH = Path(f"{PROJECT_DIR}/{uploaded_filename}")
    shutil.copy(src_path, str(PDF_PATH))
    print(f"PDF found     : {src_path}")
    print(f"PDF copied to : {PDF_PATH}")
else:
    print("WARNING: No PDF found — upload via Add Data panel then rerun")
    PDF_PATH = None
    uploaded_filename = "unknown.pdf"

# ---- Part F: Constants -------------------------------------

DOC_NAME     = uploaded_filename.replace(".pdf", "")
MIN_TEXT_LEN = 30

# Chunking — similarity is primary signal, min/max are guardrails
CHUNK_SIMILARITY_THRESHOLD = 0.4
CHUNK_MIN_CHARS            = 200
CHUNK_MAX_CHARS            = 800

# Retrieval
TOP_K_DENSE    = 20
TOP_K_SPARSE   = 20
TOP_K_GRAPH    = 10
TOP_K_FINAL    = 6
MMR_LAMBDA     = 0.6

# RAPTOR
RAPTOR_N_CLUSTERS_L1  = 5
RAPTOR_N_CLUSTERS_L2  = 3
RAPTOR_SOFT_THRESHOLD = 0.15

# Neo4j
NEO4J_URI      = "neo4j+s://YOUR_AURA_URI.databases.neo4j.io"
NEO4J_USER     = "YOUR_USERNAME"
NEO4J_PASSWORD = "YOUR_PASSWORD"


# Checkpoint paths
EXTRACTED_DOCS_PATH   = f"{PROJECT_DIR}/extracted_docs.json"
CHUNKS_PATH           = f"{PROJECT_DIR}/chunks.json"
RAPTOR_SUMMARIES_PATH = f"{PROJECT_DIR}/raptor_summaries.json"
EMBEDDINGS_PATH       = f"{PROJECT_DIR}/embeddings.npy"
QDRANT_PATH           = f"{PROJECT_DIR}/qdrant_storage"
IMAGES_DIR            = f"{PROJECT_DIR}/images"

print("\n--- All constants set ---")
print(f"DOC_NAME             : {DOC_NAME}")
print(f"PDF_PATH             : {PDF_PATH}")
print(f"Transformers version : {transformers.__version__}")
print(f"Chunk range          : {CHUNK_MIN_CHARS} – {CHUNK_MAX_CHARS} chars")
print(f"Similarity cut       : {CHUNK_SIMILARITY_THRESHOLD}")
print(f"RAPTOR L1/L2         : {RAPTOR_N_CLUSTERS_L1} / {RAPTOR_N_CLUSTERS_L2} clusters")

Transformers version : 4.44.0
Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB

Project folder : /kaggle/working/rag_project
PDF found     : /kaggle/input/datasets/yukthin1/indian-airworthy/indian-airworthy-paper1.pdf
PDF copied to : /kaggle/working/rag_project/indian-airworthy-paper1.pdf

--- All constants set ---
DOC_NAME             : indian-airworthy-paper1
PDF_PATH             : /kaggle/working/rag_project/indian-airworthy-paper1.pdf
Transformers version : 4.44.0
Chunk range          : 200 – 800 chars
Similarity cut       : 0.4
RAPTOR L1/L2         : 5 / 3 clusters


In [4]:
# ============================================================
# CELL 2 — PDF EXTRACTION
# PyMuPDF (text) + pdfplumber (tables) + placeholders (images)
# ============================================================

# ---- Part A: Build section map from PDF TOC ----------------

def build_section_map(pdf_path: str) -> Dict[int, str]:
    pdf = fitz.open(pdf_path)
    toc = pdf.get_toc()
    total_pages = len(pdf)
    pdf.close()

    section_map = {}

    if toc:
        toc_sorted = sorted(toc, key=lambda x: x[2])
        for i, (level, title, start_page) in enumerate(toc_sorted):
            end_page = toc_sorted[i+1][2] if i+1 < len(toc_sorted) else total_pages + 1
            for p in range(start_page, end_page):
                section_map[p] = title[:80]
        print(f"  TOC found — {len(toc)} entries mapped across {total_pages} pages")
    else:
        print("  No TOC found — section inferred from headings during extraction")

    return section_map


# ---- Part B: Extract text via PyMuPDF ----------------------

def extract_text(pdf_path: str, section_map: Dict[int, str]) -> List[Dict]:
    docs = []
    pdf = fitz.open(pdf_path)
    heading_pattern = re.compile(r"^(\d+[\.\d]*)\s+[A-Z]")
    current_section = "General"

    for page_num in range(len(pdf)):
        page = pdf[page_num]
        page_1indexed = page_num + 1

        if section_map:
            current_section = section_map.get(page_1indexed, current_section)

        blocks = page.get_text("blocks")
        for block_idx, b in enumerate(blocks):
            text = b[4].strip()
            if len(text) < MIN_TEXT_LEN:
                continue

            if not section_map:
                first_line = text.split("\n")[0].strip()
                if heading_pattern.match(first_line):
                    current_section = first_line[:80]

            docs.append({
                "chunk_id"    : str(uuid.uuid4()),
                "page"        : page_1indexed,
                "section"     : current_section,
                "content_type": "text",
                "doc_name"    : DOC_NAME,
                "text"        : text
            })

    pdf.close()
    return docs


# ---- Part C: Extract tables via pdfplumber -----------------

def extract_tables(pdf_path: str, section_map: Dict[int, str]) -> List[Dict]:
    docs = []
    current_section = "General"

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            page_1indexed = page_num + 1
            current_section = section_map.get(page_1indexed, current_section)

            tables = page.extract_tables()
            for t_idx, table in enumerate(tables):
                if not table:
                    continue
                rows = []
                for row in table:
                    cleaned = [cell.strip() if cell else "" for cell in row]
                    rows.append(" | ".join(cleaned))
                table_text = "\n".join(rows).strip()

                if len(table_text) < MIN_TEXT_LEN:
                    continue

                docs.append({
                    "chunk_id"    : str(uuid.uuid4()),
                    "page"        : page_1indexed,
                    "section"     : current_section,
                    "content_type": "table",
                    "doc_name"    : DOC_NAME,
                    "text"        : table_text
                })

    return docs


# ---- Part D: Extract image placeholders --------------------

def extract_images(pdf_path: str, section_map: Dict[int, str]) -> List[Dict]:
    docs = []
    pdf = fitz.open(pdf_path)
    current_section = "General"

    for page_num in range(len(pdf)):
        page = pdf[page_num]
        page_1indexed = page_num + 1
        current_section = section_map.get(page_1indexed, current_section)

        img_list = page.get_images(full=True)
        for img_idx, img in enumerate(img_list):
            docs.append({
                "chunk_id"    : str(uuid.uuid4()),
                "page"        : page_1indexed,
                "section"     : current_section,
                "content_type": "image",
                "doc_name"    : DOC_NAME,
                "text"        : f"[Image {img_idx+1} on page {page_1indexed}]"
            })

    pdf.close()
    return docs


# ---- Part E: Run full extraction ---------------------------

print("Building section map from PDF TOC...")
section_map = build_section_map(str(PDF_PATH))

print("\nExtracting text blocks...")
text_docs = extract_text(str(PDF_PATH), section_map)
print(f"  Text blocks extracted : {len(text_docs)}")

print("\nExtracting tables...")
table_docs = extract_tables(str(PDF_PATH), section_map)
print(f"  Tables extracted      : {len(table_docs)}")

print("\nExtracting image placeholders...")
image_docs = extract_images(str(PDF_PATH), section_map)
print(f"  Image placeholders    : {len(image_docs)}")

all_docs = text_docs + table_docs + image_docs
print(f"\nTotal documents extracted : {len(all_docs)}")

# ---- Part F: Save checkpoint -------------------------------

with open(EXTRACTED_DOCS_PATH, "w") as f:
    json.dump(all_docs, f, indent=2)
print(f"\nCheckpoint saved → {EXTRACTED_DOCS_PATH}")

# ---- Part G: Summary ---------------------------------------

content_counts = {}
for d in all_docs:
    ct = d["content_type"]
    content_counts[ct] = content_counts.get(ct, 0) + 1

print("\n--- Extraction complete ---")
for ct, count in content_counts.items():
    print(f"  {ct:<10} : {count}")

print("\nSample text block:")
sample = next((d for d in all_docs if d["content_type"] == "text"), None)
if sample:
    print(f"  chunk_id : {sample['chunk_id']}")
    print(f"  page     : {sample['page']}")
    print(f"  section  : {sample['section']}")
    print(f"  text     : {sample['text'][:200]}")

Building section map from PDF TOC...
  No TOC found — section inferred from headings during extraction

Extracting text blocks...
  Text blocks extracted : 970

Extracting tables...
  Tables extracted      : 4

Extracting image placeholders...
  Image placeholders    : 3

Total documents extracted : 977

Checkpoint saved → /kaggle/working/rag_project/extracted_docs.json

--- Extraction complete ---
  text       : 970
  table      : 4
  image      : 3

Sample text block:
  chunk_id : 99f061b3-625b-4dc6-9589-78879335b037
  page     : 1
  section  : General
  text     : INDIAN MILITARY 
AIRWORTHINESS
PROCEDURE - 2023
IMAP-2023


In [5]:
# ============================================================
# CELL 3 — PII SCAN
# ============================================================

print("Running PII scan across all extracted documents...\n")

pii_patterns = {
    "email"  : re.compile(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+"),
    "phone"  : re.compile(r"\b(\+91[\-\s]?)?[6-9]\d{9}\b"),
    "aadhaar": re.compile(r"\b\d{4}\s\d{4}\s\d{4}\b"),
    "pan"    : re.compile(r"\b[A-Z]{5}[0-9]{4}[A-Z]\b"),
    "ip"     : re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b"),
}

findings = []

for doc in all_docs:
    text = doc.get("text", "")
    for pii_type, pattern in pii_patterns.items():
        matches = pattern.findall(text)
        if matches:
            findings.append({
                "chunk_id"    : doc["chunk_id"],
                "page"        : doc["page"],
                "section"     : doc["section"],
                "content_type": doc["content_type"],
                "pii_type"    : pii_type,
                "matches"     : matches
            })

print(f"Documents scanned : {len(all_docs)}")
print(f"PII findings      : {len(findings)}")

if findings:
    print("\nWARNING — PII detected:\n")
    for f in findings:
        print(f"  chunk_id : {f['chunk_id']}")
        print(f"  page     : {f['page']}")
        print(f"  type     : {f['pii_type']}")
        print(f"  matches  : {f['matches']}")
        print()
    print("ACTION REQUIRED: Review before indexing.")
else:
    print("\nDocument is clean — no PII detected.")

print("\n--- PII scan complete ---")

Running PII scan across all extracted documents...

Documents scanned : 977
PII findings      : 0

Document is clean — no PII detected.

--- PII scan complete ---


In [6]:
# ============================================================
# CELL 4 — MODEL LOADING
# BGE-small, Cross-encoder, NLI, Phi-3.5-mini
# ============================================================

print("Loading models...\n")

# ---- 1. BGE-small — Embeddings -----------------------------

print("Loading BGE-small embedding model...")
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)
embed_model.eval()
print(f"  BGE-small loaded — embedding dim : {embed_model.get_embedding_dimension()}")

# ---- 2. Cross-encoder — Reranking --------------------------

print("\nLoading cross-encoder reranker...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)
print("  Cross-encoder loaded")

# ---- 3. NLI model — Evaluation -----------------------------

print("\nLoading NLI model for evaluation...")
nli_pipeline = pipeline(
    "text-classification",
    model="cross-encoder/nli-deberta-v3-small",
    device=0 if device == "cuda" else -1
)
print("  NLI model loaded")

# ---- 4. Phi-3.5-mini ---------------------------------------

print("\nLoading Phi-3.5-mini in float16...")
phi_tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/Phi-3.5-mini-instruct",
    trust_remote_code=True
)
phi_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3.5-mini-instruct",
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
phi_model.eval()
print("  Phi-3.5-mini loaded")

# ---- 5. Helper — Phi-3.5-mini inference --------------------

def run_phi3(prompt: str, max_new_tokens: int = 512) -> str:
    messages = [{"role": "user", "content": prompt}]

    tokenized = phi_tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )
    input_ids      = tokenized["input_ids"].to(phi_model.device)
    attention_mask = tokenized["attention_mask"].to(phi_model.device)

    with torch.no_grad():
        output_ids = phi_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=phi_tokenizer.eos_token_id
        )

    generated = output_ids[0][input_ids.shape[-1]:]
    return phi_tokenizer.decode(generated, skip_special_tokens=True).strip()


# ---- 6. Helper — BGE embeddings ----------------------------

def get_embeddings(texts: List[str], batch_size: int = 32) -> np.ndarray:
    embeddings = embed_model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True
    )
    return embeddings


# ---- 7. Sanity checks --------------------------------------

print("\nRunning sanity checks...")

# Embedding test
test_emb = get_embeddings(["test sentence"], batch_size=1)
assert test_emb.shape == (1, 384), f"Unexpected shape: {test_emb.shape}"
print(f"  Embedding shape : {test_emb.shape} ✓")

# Generation test — real domain prompt
test_response = run_phi3(
    "What is airworthiness? Answer in one sentence.",
    max_new_tokens=60
)
print(f"  Phi-3.5-mini    : {test_response}")

# ---- 8. VRAM check -----------------------------------------

if device == "cuda":
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved  = torch.cuda.memory_reserved(0) / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    free      = total - reserved
    print(f"\nVRAM after model loading:")
    print(f"  Allocated : {allocated:.2f} GB")
    print(f"  Reserved  : {reserved:.2f} GB")
    print(f"  Free      : {free:.2f} GB remaining")

print("\n--- All models loaded and ready ---")

Loading models...

Loading BGE-small embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


  Embedding shape : (1, 384) ✓


You are not running the flash-attention implementation, expect numerical differences.


  Phi-3.5-mini    : Airworthiness is the measure of an aircraft's suitability for safe flight, determined by its compliance with established safety and design standards.

VRAM after model loading:
  Allocated : 4.62 GB
  Reserved  : 4.66 GB
  Free      : 10.97 GB remaining

--- All models loaded and ready ---


In [7]:
# ============================================================
# CELL 5 — SEMANTIC CHUNKING
# Cosine similarity based topic-shift detection
# Short doc merging before chunking
# Tables and images kept as single chunks
# ============================================================

# ---- Part A: Merge short consecutive docs ------------------

def merge_short_docs(docs: List[Dict]) -> List[Dict]:
    """
    Merge consecutive short text blocks from the same page
    and section before chunking.
    Prevents tiny fragments entering the chunker.
    """
    merged = []
    buffer = None

    for doc in docs:
        # Non-text docs pass through unchanged
        if doc["content_type"] != "text":
            if buffer:
                merged.append(buffer)
                buffer = None
            merged.append(doc)
            continue

        if buffer is None:
            buffer = doc.copy()
        else:
            same_page    = buffer["page"] == doc["page"]
            same_section = buffer["section"] == doc["section"]
            buffer_short = len(buffer["text"]) < CHUNK_MIN_CHARS

            if same_page and same_section and buffer_short:
                # Merge into buffer
                buffer["text"]       = buffer["text"] + " " + doc["text"]
                buffer["char_count"] = len(buffer["text"])
            else:
                merged.append(buffer)
                buffer = doc.copy()

    if buffer:
        merged.append(buffer)

    return merged


# ---- Part B: Semantic chunker for one document -------------

def semantic_chunk_text(doc: Dict) -> List[Dict]:
    """
    Split a single text document into semantic chunks.
    Primary signal  — cosine similarity drop between sentences
    Guardrail min   — CHUNK_MIN_CHARS = 200
    Guardrail max   — CHUNK_MAX_CHARS = 800
    """
    text      = doc["text"]
    sentences = sent_tokenize(text)

    # Single sentence or already short — return as is
    if len(sentences) <= 1 or len(text) <= CHUNK_MIN_CHARS:
        if len(text) >= 20:
            return [{
                "chunk_id"    : str(uuid.uuid4()),
                "text"        : text.strip(),
                "page"        : doc["page"],
                "section"     : doc["section"],
                "content_type": doc["content_type"],
                "doc_name"    : doc["doc_name"],
                "char_count"  : len(text)
            }]
        else:
            return []

    # Embed all sentences in one batch
    sentence_embeddings = get_embeddings(sentences, batch_size=64)

    # Cosine similarity between consecutive sentences
    # BGE vectors are normalized — dot product = cosine similarity
    similarities = []
    for i in range(len(sentence_embeddings) - 1):
        sim = float(np.dot(sentence_embeddings[i], sentence_embeddings[i+1]))
        similarities.append(sim)

    # Walk sentences and cut at topic shifts
    chunks  = []
    current = [sentences[0]]

    for i, sim in enumerate(similarities):
        next_sentence = sentences[i+1]
        current_text  = " ".join(current)
        projected     = current_text + " " + next_sentence

        if len(projected) > CHUNK_MAX_CHARS:
            # Force cut — projected chunk too long
            if len(current_text) >= CHUNK_MIN_CHARS:
                chunks.append(current_text)
                current = [next_sentence]
            else:
                # Below min — absorb anyway
                current.append(next_sentence)

        elif sim < CHUNK_SIMILARITY_THRESHOLD:
            # Topic shift detected
            if len(current_text) >= CHUNK_MIN_CHARS:
                chunks.append(current_text)
                current = [next_sentence]
            else:
                # Too small to cut — keep building
                current.append(next_sentence)

        else:
            # No shift — keep building
            current.append(next_sentence)

    # Handle last chunk
    if current:
        last = " ".join(current).strip()
        if chunks and len(last) < CHUNK_MIN_CHARS:
            # Merge tiny tail into previous chunk
            chunks[-1] = chunks[-1] + " " + last
        else:
            chunks.append(last)

    # Convert to chunk dicts with full metadata
    result = []
    for chunk_text in chunks:
        chunk_text = chunk_text.strip()
        if len(chunk_text) < 20:
            continue
        result.append({
            "chunk_id"    : str(uuid.uuid4()),
            "text"        : chunk_text,
            "page"        : doc["page"],
            "section"     : doc["section"],
            "content_type": doc["content_type"],
            "doc_name"    : doc["doc_name"],
            "char_count"  : len(chunk_text)
        })

    return result


# ---- Part C: Process all docs ------------------------------

def process_all_docs(all_docs: List[Dict]) -> List[Dict]:
    """
    Full chunking pipeline:
    1. Merge short consecutive text blocks
    2. Semantic chunk text docs
    3. Pass tables and images through as single chunks
    """
    all_chunks = []

    # Step 1 — merge short docs
    print("Merging short consecutive text blocks...")
    merged_docs = merge_short_docs(all_docs)

    text_docs  = [d for d in merged_docs if d["content_type"] == "text"]
    table_docs = [d for d in merged_docs if d["content_type"] == "table"]
    image_docs = [d for d in merged_docs if d["content_type"] == "image"]

    print(f"  Docs before merge : {len(all_docs)}")
    print(f"  Docs after merge  : {len(merged_docs)}")
    print(f"  Text docs         : {len(text_docs)}")

    # Step 2 — semantic chunk text docs
    print(f"\nSemantic chunking {len(text_docs)} text documents...")
    for doc in tqdm(text_docs, desc="Chunking"):
        chunks = semantic_chunk_text(doc)
        all_chunks.extend(chunks)

    # Step 3 — tables as single chunks
    print(f"\nKeeping {len(table_docs)} tables as single chunks...")
    for doc in table_docs:
        all_chunks.append({
            "chunk_id"    : str(uuid.uuid4()),
            "text"        : doc["text"],
            "page"        : doc["page"],
            "section"     : doc["section"],
            "content_type": doc["content_type"],
            "doc_name"    : doc["doc_name"],
            "char_count"  : len(doc["text"])
        })

    # Step 4 — images as single chunks
    print(f"Keeping {len(image_docs)} image placeholders as single chunks...")
    for doc in image_docs:
        all_chunks.append({
            "chunk_id"    : str(uuid.uuid4()),
            "text"        : doc["text"],
            "page"        : doc["page"],
            "section"     : doc["section"],
            "content_type": doc["content_type"],
            "doc_name"    : doc["doc_name"],
            "char_count"  : len(doc["text"])
        })

    return all_chunks


# ---- Part D: Run -------------------------------------------

print("=" * 50)
print("CELL 5 — SEMANTIC CHUNKING")
print("=" * 50 + "\n")

all_chunks = process_all_docs(all_docs)

# ---- Part E: Statistics ------------------------------------

text_chunks  = [c for c in all_chunks if c["content_type"] == "text"]
table_chunks = [c for c in all_chunks if c["content_type"] == "table"]
image_chunks = [c for c in all_chunks if c["content_type"] == "image"]
char_counts  = [c["char_count"] for c in text_chunks]

print(f"\n--- Chunking complete ---")
print(f"  Total chunks   : {len(all_chunks)}")
print(f"  Text chunks    : {len(text_chunks)}")
print(f"  Table chunks   : {len(table_chunks)}")
print(f"  Image chunks   : {len(image_chunks)}")

if char_counts:
    print(f"\n  Text chunk size distribution:")
    print(f"    Min    : {min(char_counts)} chars")
    print(f"    Max    : {max(char_counts)} chars")
    print(f"    Mean   : {int(np.mean(char_counts))} chars")
    print(f"    Median : {int(np.median(char_counts))} chars")

    # Distribution buckets
    under_200  = sum(1 for c in char_counts if c < 200)
    b200_400   = sum(1 for c in char_counts if 200 <= c < 400)
    b400_600   = sum(1 for c in char_counts if 400 <= c < 600)
    b600_800   = sum(1 for c in char_counts if 600 <= c < 800)
    over_800   = sum(1 for c in char_counts if c >= 800)

    print(f"\n  Size distribution:")
    print(f"    < 200 chars  : {under_200}")
    print(f"    200-400      : {b200_400}")
    print(f"    400-600      : {b400_600}")
    print(f"    600-800      : {b600_800}")
    print(f"    > 800 chars  : {over_800}")

# ---- Part F: Sample chunks ---------------------------------

print("\nSample chunk 1 (text):")
sample = next((c for c in all_chunks if c["content_type"] == "text"
               and c["char_count"] > 200), None)
if sample:
    print(f"  chunk_id   : {sample['chunk_id']}")
    print(f"  page       : {sample['page']}")
    print(f"  section    : {sample['section']}")
    print(f"  char_count : {sample['char_count']}")
    print(f"  text       : {sample['text'][:300]}")

print("\nSample chunk 2 (table):")
sample_t = next((c for c in all_chunks if c["content_type"] == "table"), None)
if sample_t:
    print(f"  chunk_id   : {sample_t['chunk_id']}")
    print(f"  page       : {sample_t['page']}")
    print(f"  char_count : {sample_t['char_count']}")
    print(f"  text       : {sample_t['text'][:200]}")

# ---- Part G: Save checkpoint -------------------------------

with open(CHUNKS_PATH, "w") as f:
    json.dump(all_chunks, f, indent=2)
print(f"\nCheckpoint saved → {CHUNKS_PATH}")

CELL 5 — SEMANTIC CHUNKING

Merging short consecutive text blocks...
  Docs before merge : 977
  Docs after merge  : 563
  Text docs         : 556

Semantic chunking 556 text documents...


Chunking:   0%|          | 0/556 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Chunking:   1%|          | 4/556 [00:00<00:16, 33.77it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Chunking:  99%|█████████▉| 552/556 [00:09<00:00, 60.64it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Chunking: 100%|██████████| 556/556 [00:09<00:00, 57.75it/s]



Keeping 4 tables as single chunks...
Keeping 3 image placeholders as single chunks...

--- Chunking complete ---
  Total chunks   : 569
  Text chunks    : 562
  Table chunks   : 4
  Image chunks   : 3

  Text chunk size distribution:
    Min    : 33 chars
    Max    : 2204 chars
    Mean   : 354 chars
    Median : 314 chars

  Size distribution:
    < 200 chars  : 58
    200-400      : 350
    400-600      : 105
    600-800      : 37
    > 800 chars  : 12

Sample chunk 1 (text):
  chunk_id   : 1c506565-96dc-43e9-841b-fce2c5684044
  page       : 2
  section    : General
  char_count : 216
  text       : Indian Military 
Airworthiness Procedure - 2023 
(IMAP-2023) Suggestions for improvement of this document should be addressed to:- Joint Airworthiness Committee (JAC) Centre for Military Airworthiness & Certification

Sample chunk 2 (table):
  chunk_id   : 9a6e5c57-8a8a-45c3-a947-23d039a888a8
  page       : 5
  char_count : 181
  text       : Sl.
No. | Amendment
Number | Date of
Amendme

In [8]:
# ============================================================
# CELL 6 — EMBEDDINGS
# BGE-small dense vectors + true BM25 sparse vectors
# ============================================================

print("=" * 50)
print("CELL 6 — EMBEDDINGS")
print("=" * 50 + "\n")

# ---- Part A: Extract texts and chunk_ids -------------------

texts     = [c["text"]     for c in all_chunks]
chunk_ids = [c["chunk_id"] for c in all_chunks]

print(f"Total chunks to embed : {len(texts)}")

# ---- Part B: Dense vectors — BGE-small ---------------------

print("\nComputing BGE-small dense vectors...")
dense_vectors = get_embeddings(texts, batch_size=64)

print(f"  Dense vectors shape : {dense_vectors.shape}")
print(f"  Sample norm         : {np.linalg.norm(dense_vectors[0]):.4f} (should be ~1.0)")

# Save embeddings to disk
np.save(EMBEDDINGS_PATH, dense_vectors)
print(f"  Saved → {EMBEDDINGS_PATH}")

# ---- Part C: Sparse vectors — true BM25 TF×IDF ------------

print("\nComputing true BM25 sparse vectors...")

# Tokenize all chunks
tokenized_corpus = []
for text in texts:
    tokens = re.findall(r'\b[a-zA-Z]{2,}\b', text.lower())
    tokenized_corpus.append(tokens)

# Build BM25 model over full corpus
bm25 = BM25Okapi(tokenized_corpus)

# For each chunk compute full BM25 score per token
# BM25Okapi stores idf scores — we compute TF component manually
sparse_vectors = []

for idx, tokens in enumerate(tqdm(tokenized_corpus, desc="BM25")):
    if not tokens:
        sparse_vectors.append({})
        continue

    # Count term frequencies in this chunk
    tf_counts = {}
    for token in tokens:
        tf_counts[token] = tf_counts.get(token, 0) + 1

    # Get unique tokens
    unique_tokens = list(tf_counts.keys())

    # Compute BM25 score for each unique token
    # BM25Okapi.get_scores returns scores for a query against all docs
    # We use a single-token query to get IDF component
    # Then combine with TF using BM25 formula
    k1  = bm25.k1   # default 1.5
    b   = bm25.b     # default 0.75
    avgdl = bm25.avgdl
    dl  = len(tokens)

    sparse_vec = {}
    for token in unique_tokens:
        if token not in bm25.idf:
            continue
        idf = bm25.idf[token]
        tf  = tf_counts[token]

        # Full BM25Okapi formula
        # score = IDF × (TF × (k1 + 1)) / (TF + k1 × (1 - b + b × dl/avgdl))
        numerator   = tf * (k1 + 1)
        denominator = tf + k1 * (1 - b + b * (dl / avgdl))
        score       = idf * (numerator / denominator)

        if score > 0:
            sparse_vec[token] = round(float(score), 4)

    sparse_vectors.append(sparse_vec)

# Verify
non_empty = sum(1 for sv in sparse_vectors if sv)
avg_terms = np.mean([len(sv) for sv in sparse_vectors if sv])
print(f"  Non-empty sparse vectors : {non_empty} / {len(sparse_vectors)}")
print(f"  Avg unique terms/chunk   : {avg_terms:.1f}")

# Sample sparse vector
sample_idx = next(i for i, sv in enumerate(sparse_vectors) if sv)
sample_sv  = dict(sorted(
    sparse_vectors[sample_idx].items(),
    key=lambda x: x[1],
    reverse=True
)[:10])
print(f"\n  Sample sparse vector (top 10 terms):")
for term, score in sample_sv.items():
    print(f"    {term:<20} : {score:.4f}")

# ---- Part D: Build combined embedding store ----------------

print("\nBuilding combined embedding store...")
embedding_store = {}
for i, chunk_id in enumerate(chunk_ids):
    embedding_store[chunk_id] = {
        "dense_vector" : dense_vectors[i].tolist(),
        "sparse_vector": sparse_vectors[i]
    }

print(f"  Embedding store size : {len(embedding_store)} entries")

# ---- Part E: Save checkpoint -------------------------------

embedding_store_path = f"{PROJECT_DIR}/embedding_store.json"
with open(embedding_store_path, "w") as f:
    json.dump(embedding_store, f)
print(f"  Saved → {embedding_store_path}")

# ---- Part F: Summary ---------------------------------------

print(f"\n--- Embeddings complete ---")
print(f"  Chunks embedded      : {len(chunk_ids)}")
print(f"  Dense vector dim     : {dense_vectors.shape[1]}")
print(f"  Sparse vocab size    : {len(bm25.idf)}")
print(f"  Sample norm check    : {np.linalg.norm(dense_vectors[0]):.4f}")

CELL 6 — EMBEDDINGS

Total chunks to embed : 569

Computing BGE-small dense vectors...


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

  Dense vectors shape : (569, 384)
  Sample norm         : 1.0000 (should be ~1.0)
  Saved → /kaggle/working/rag_project/embeddings.npy

Computing true BM25 sparse vectors...


BM25: 100%|██████████| 569/569 [00:00<00:00, 13121.18it/s]

  Non-empty sparse vectors : 569 / 569
  Avg unique terms/chunk   : 34.9

  Sample sparse vector (top 10 terms):
    supersedes           : 8.0983
    dated                : 8.0983
    feb                  : 8.0983
    ddpmas               : 7.3991
    version              : 6.9378
    imap                 : 5.2959
    ministry             : 4.8790
    government           : 4.8790
    framework            : 4.6303
    procedure            : 3.9302

Building combined embedding store...
  Embedding store size : 569 entries


  Saved → /kaggle/working/rag_project/embedding_store.json

--- Embeddings complete ---
  Chunks embedded      : 569
  Dense vector dim     : 384
  Sparse vocab size    : 2074
  Sample norm check    : 1.0000


In [9]:
# ============================================================
# CELL 7 — QDRANT PERSISTENT INDEXING
# Dense + sparse named vectors + full metadata payload
# ============================================================

print("=" * 50)
print("CELL 7 — QDRANT INDEXING")
print("=" * 50 + "\n")

# ---- Part A: Close any existing client ---------------------

try:
    qdrant.close()
    print("Previous Qdrant client closed")
except:
    pass

# ---- Part B: Initialize persistent Qdrant client ----------

print(f"Initializing Qdrant at : {QDRANT_PATH}")
qdrant = QdrantClient(path=QDRANT_PATH)

COLLECTION_NAME = "imap_rag"

# Delete collection if exists — clean slate
existing = [c.name for c in qdrant.get_collections().collections]
if COLLECTION_NAME in existing:
    qdrant.delete_collection(COLLECTION_NAME)
    print(f"  Deleted existing collection : {COLLECTION_NAME}")

# Create collection with dense + sparse named vectors
qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "dense": VectorParams(
            size=384,
            distance=Distance.COSINE
        )
    },
    sparse_vectors_config={
        "sparse": SparseVectorParams()
    }
)
print(f"  Collection created : {COLLECTION_NAME}")

# ---- Part C: Build Qdrant points ---------------------------

print("\nBuilding Qdrant points...")
points = []

for chunk in tqdm(all_chunks, desc="Building points"):
    chunk_id  = chunk["chunk_id"]
    store     = embedding_store.get(chunk_id)

    if store is None:
        continue

    dense_vec  = store["dense_vector"]
    sparse_vec = store["sparse_vector"]

    if sparse_vec:
        sparse_indices = []
        sparse_values  = []
        for token, score in sparse_vec.items():
            sparse_indices.append(abs(hash(token)) % (10**6))
            sparse_values.append(float(score))
    else:
        sparse_indices = [0]
        sparse_values  = [0.0]

    payload = {
        "chunk_id"    : chunk_id,
        "text"        : chunk["text"],
        "page"        : chunk["page"],
        "section"     : chunk["section"],
        "content_type": chunk["content_type"],
        "doc_name"    : chunk["doc_name"],
        "char_count"  : chunk["char_count"]
    }

    point = PointStruct(
        id      = abs(hash(chunk_id)) % (10**12),
        payload = payload,
        vector  = {
            "dense" : dense_vec,
            "sparse": SparseVector(
                indices=sparse_indices,
                values=sparse_values
            )
        }
    )
    points.append(point)

print(f"  Points built : {len(points)}")

# ---- Part D: Upload points in batches ----------------------

print("\nUploading points to Qdrant...")
BATCH_SIZE = 100

for i in tqdm(range(0, len(points), BATCH_SIZE), desc="Uploading"):
    batch = points[i:i + BATCH_SIZE]
    qdrant.upsert(
        collection_name=COLLECTION_NAME,
        points=batch
    )

# ---- Part E: Verify ----------------------------------------

collection_info = qdrant.get_collection(COLLECTION_NAME)
indexed_count   = collection_info.points_count

print(f"\n--- Qdrant indexing complete ---")
print(f"  Collection      : {COLLECTION_NAME}")
print(f"  Points indexed  : {indexed_count}")
print(f"  Storage path    : {QDRANT_PATH}")

# ---- Part F: Quick retrieval test --------------------------

print("\nRunning quick retrieval test...")
test_query     = "airworthiness certification procedure"
test_embedding = get_embeddings([test_query], batch_size=1)[0].tolist()

results = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    query=test_embedding,
    using="dense",
    limit=3
).points

print(f"  Query : '{test_query}'")
print(f"  Top 3 results:")
for i, r in enumerate(results):
    print(f"\n  [{i+1}] score : {r.score:.4f}")
    print(f"       page    : {r.payload['page']}")
    print(f"       section : {r.payload['section']}")
    print(f"       text    : {r.payload['text'][:150]}")

# ---- Part G: Save ID map -----------------------------------

id_map = {
    chunk["chunk_id"]: abs(hash(chunk["chunk_id"])) % (10**12)
    for chunk in all_chunks
}
id_map_path = f"{PROJECT_DIR}/chunk_id_map.json"
with open(id_map_path, "w") as f:
    json.dump(id_map, f)
print(f"\nID map saved → {id_map_path}")

CELL 7 — QDRANT INDEXING

Initializing Qdrant at : /kaggle/working/rag_project/qdrant_storage
  Collection created : imap_rag

Building Qdrant points...


Building points: 100%|██████████| 569/569 [00:00<00:00, 22319.93it/s]


  Points built : 569

Uploading points to Qdrant...


Uploading: 100%|██████████| 6/6 [00:04<00:00,  1.35it/s]


--- Qdrant indexing complete ---
  Collection      : imap_rag
  Points indexed  : 569
  Storage path    : /kaggle/working/rag_project/qdrant_storage

Running quick retrieval test...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Query : 'airworthiness certification procedure'
  Top 3 results:

  [1] score : 0.8336
       page    : 31
       section : 1.4.2  Airworthiness Certification Criteria
       text    : 1.4.2  Airworthiness Certification Criteria a. Main Contractor shall ensure that the Air System is designed to an applicable 
Airworthiness Certificat

  [2] score : 0.8248
       page    : 25
       section : 3.3	 Ensuring Airworthiness Requirements during Acquisition
       text    : a. Any product procured for use in Airborne applications shall have an airworthiness 
certificate. The airworthiness requirements detailed in subseque

  [3] score : 0.8200
       page    : 40
       section : 1.5.2	Airworthiness Certification Criteria
       text    : 1.5.2	Airworthiness Certification Criteria a. Main Contractor shall ensure that the Airborne Stores is designed to an applicable 
Airworthiness Certif

ID map saved → /kaggle/working/rag_project/chunk_id_map.json


In [10]:
# ============================================================
# CELL 8 — RAPTOR
# Two-level hierarchical summarisation
# Soft GMM assignments + Phi-3.5-mini summaries
# Summary nodes added back into Qdrant
# ============================================================

print("=" * 50)
print("CELL 8 — RAPTOR")
print("=" * 50 + "\n")

# ---- Part A: Prepare chunk embeddings ----------------------

print("Preparing chunk embeddings for clustering...")

# Use only text chunks for RAPTOR — tables and images excluded
text_chunk_indices = [
    i for i, c in enumerate(all_chunks)
    if c["content_type"] == "text"
]
text_chunks_only = [all_chunks[i] for i in text_chunk_indices]
text_embeddings  = dense_vectors[text_chunk_indices]

print(f"  Text chunks for RAPTOR : {len(text_chunks_only)}")
print(f"  Embedding matrix shape : {text_embeddings.shape}")

# ---- Part B: PCA dimensionality reduction ------------------

print("\nReducing dimensions with PCA...")
N_COMPONENTS = min(50, text_embeddings.shape[0] - 1, text_embeddings.shape[1])
pca          = PCA(n_components=N_COMPONENTS)
reduced      = pca.fit_transform(text_embeddings)
variance     = sum(pca.explained_variance_ratio_) * 100
print(f"  Reduced : 384 → {N_COMPONENTS} dims")
print(f"  Variance explained : {variance:.1f}%")

# ---- Part C: Level 1 — Soft GMM clustering -----------------

print(f"\nLevel 1 — GMM clustering into {RAPTOR_N_CLUSTERS_L1} clusters...")
gmm_l1 = GaussianMixture(
    n_components=RAPTOR_N_CLUSTERS_L1,
    covariance_type="full",
    random_state=42
)
gmm_l1.fit(reduced)

# Soft assignments — probability per cluster per chunk
probs_l1 = gmm_l1.predict_proba(reduced)

# Assign each chunk to every cluster where prob > threshold
cluster_assignments_l1 = {i: [] for i in range(RAPTOR_N_CLUSTERS_L1)}
for chunk_idx, probs in enumerate(probs_l1):
    for cluster_idx, prob in enumerate(probs):
        if prob >= RAPTOR_SOFT_THRESHOLD:
            cluster_assignments_l1[cluster_idx].append(chunk_idx)

for c_idx, members in cluster_assignments_l1.items():
    print(f"  Cluster {c_idx} : {len(members)} chunks")

# ---- Part D: Level 1 — Generate summaries ------------------

print("\nGenerating Level 1 summaries with Phi-3.5-mini...")
l1_summaries = []

for cluster_idx, member_indices in cluster_assignments_l1.items():
    if not member_indices:
        continue

    # Gather member texts — truncate each to 300 chars to fit context
    member_texts = []
    for idx in member_indices[:20]:  # cap at 20 chunks per cluster
        chunk = text_chunks_only[idx]
        member_texts.append(chunk["text"][:300])

    combined = "\n\n".join(member_texts)

    prompt = f"""You are summarising a cluster of text chunks from an Indian Military Airworthiness document (IMAP-2023).

Chunks:
{combined}

Write a concise summary (3-5 sentences) that captures the key topics, procedures, and regulatory concepts covered in these chunks. Be specific — mention actual procedures, bodies, or requirements if present."""

    print(f"  Summarising cluster {cluster_idx} ({len(member_indices)} chunks)...")
    summary_text = run_phi3(prompt, max_new_tokens=200)

    # Get representative page and section from most central chunk
    central_idx     = member_indices[0]
    central_chunk   = text_chunks_only[central_idx]

    l1_summaries.append({
        "chunk_id"      : str(uuid.uuid4()),
        "text"          : summary_text,
        "page"          : central_chunk["page"],
        "section"       : central_chunk["section"],
        "content_type"  : "raptor_l1_summary",
        "doc_name"      : DOC_NAME,
        "char_count"    : len(summary_text),
        "cluster_idx"   : cluster_idx,
        "member_count"  : len(member_indices),
        "raptor_level"  : 1
    })
    print(f"    Summary : {summary_text[:150]}...")

# ---- Part E: Level 2 — Cluster the L1 summaries -----------

print(f"\nLevel 2 — Clustering {len(l1_summaries)} L1 summaries into {RAPTOR_N_CLUSTERS_L2} meta-clusters...")

l1_texts      = [s["text"] for s in l1_summaries]
l1_embeddings = get_embeddings(l1_texts, batch_size=8)

# PCA on L1 summaries
N_COMPONENTS_L2 = min(RAPTOR_N_CLUSTERS_L2 + 1, len(l1_summaries) - 1)
pca_l2          = PCA(n_components=N_COMPONENTS_L2)
reduced_l2      = pca_l2.fit_transform(l1_embeddings)

gmm_l2 = GaussianMixture(
    n_components=min(RAPTOR_N_CLUSTERS_L2, len(l1_summaries)),
    covariance_type="full",
    random_state=42
)
gmm_l2.fit(reduced_l2)
probs_l2 = gmm_l2.predict_proba(reduced_l2)

cluster_assignments_l2 = {i: [] for i in range(RAPTOR_N_CLUSTERS_L2)}
for summary_idx, probs in enumerate(probs_l2):
    for cluster_idx, prob in enumerate(probs):
        if prob >= RAPTOR_SOFT_THRESHOLD:
            cluster_assignments_l2[cluster_idx].append(summary_idx)

for c_idx, members in cluster_assignments_l2.items():
    print(f"  Meta-cluster {c_idx} : {len(members)} L1 summaries")

# ---- Part F: Level 2 — Generate meta-summaries -------------

print("\nGenerating Level 2 meta-summaries with Phi-3.5-mini...")
l2_summaries = []

for cluster_idx, member_indices in cluster_assignments_l2.items():
    if not member_indices:
        continue

    member_texts = [l1_summaries[idx]["text"] for idx in member_indices]
    combined     = "\n\n".join(member_texts)

    prompt = f"""You are creating a high-level meta-summary from cluster summaries of an Indian Military Airworthiness document (IMAP-2023).

Cluster summaries:
{combined}

Write a high-level meta-summary (3-5 sentences) that captures the overarching themes, regulatory framework, and key processes across these clusters."""

    print(f"  Summarising meta-cluster {cluster_idx}...")
    meta_summary = run_phi3(prompt, max_new_tokens=200)

    l2_summaries.append({
        "chunk_id"    : str(uuid.uuid4()),
        "text"        : meta_summary,
        "page"        : 0,
        "section"     : "RAPTOR Meta-Summary",
        "content_type": "raptor_l2_summary",
        "doc_name"    : DOC_NAME,
        "char_count"  : len(meta_summary),
        "cluster_idx" : cluster_idx,
        "raptor_level": 2
    })
    print(f"    Meta-summary : {meta_summary[:150]}...")

# ---- Part G: Add summary nodes to Qdrant ------------------

print("\nAdding RAPTOR summary nodes to Qdrant...")
all_summaries = l1_summaries + l2_summaries

summary_texts      = [s["text"] for s in all_summaries]
summary_embeddings = get_embeddings(summary_texts, batch_size=8)

summary_points = []
for i, summary in enumerate(all_summaries):
    dense_vec = summary_embeddings[i].tolist()

    # Sparse vector for summary
    tokens = re.findall(r'\b[a-zA-Z]{2,}\b', summary["text"].lower())
    tf_counts = {}
    for token in tokens:
        tf_counts[token] = tf_counts.get(token, 0) + 1

    sparse_indices = []
    sparse_values  = []
    for token, tf in tf_counts.items():
        if token in bm25.idf:
            sparse_indices.append(abs(hash(token)) % (10**6))
            sparse_values.append(float(tf * bm25.idf[token]))

    if not sparse_indices:
        sparse_indices = [0]
        sparse_values  = [0.0]

    payload = {
        "chunk_id"    : summary["chunk_id"],
        "text"        : summary["text"],
        "page"        : summary["page"],
        "section"     : summary["section"],
        "content_type": summary["content_type"],
        "doc_name"    : summary["doc_name"],
        "char_count"  : summary["char_count"],
        "raptor_level": summary["raptor_level"]
    }

    summary_points.append(PointStruct(
        id      = abs(hash(summary["chunk_id"])) % (10**12),
        payload = payload,
        vector  = {
            "dense" : dense_vec,
            "sparse": SparseVector(
                indices=sparse_indices,
                values=sparse_values
            )
        }
    ))

qdrant.upsert(
    collection_name=COLLECTION_NAME,
    points=summary_points
)

total_points = qdrant.get_collection(COLLECTION_NAME).points_count
print(f"  Summary nodes added   : {len(summary_points)}")
print(f"  Total points in Qdrant: {total_points}")

# ---- Part H: Save checkpoint -------------------------------

all_raptor = {
    "l1_summaries": l1_summaries,
    "l2_summaries": l2_summaries
}
with open(RAPTOR_SUMMARIES_PATH, "w") as f:
    json.dump(all_raptor, f, indent=2)
print(f"\nCheckpoint saved → {RAPTOR_SUMMARIES_PATH}")

# ---- Part I: Summary ---------------------------------------

print("\n--- RAPTOR complete ---")
print(f"  L1 summaries   : {len(l1_summaries)}")
print(f"  L2 summaries   : {len(l2_summaries)}")
print(f"  Total summaries: {len(all_summaries)}")
print(f"  Qdrant total   : {total_points} points")
print(f"\nSample L1 summary:")
if l1_summaries:
    print(f"  {l1_summaries[0]['text'][:300]}")
print(f"\nSample L2 meta-summary:")
if l2_summaries:
    print(f"  {l2_summaries[0]['text'][:300]}")

CELL 8 — RAPTOR

Preparing chunk embeddings for clustering...
  Text chunks for RAPTOR : 562
  Embedding matrix shape : (562, 384)

Reducing dimensions with PCA...
  Reduced : 384 → 50 dims
  Variance explained : 72.9%

Level 1 — GMM clustering into 5 clusters...
  Cluster 0 : 89 chunks
  Cluster 1 : 99 chunks
  Cluster 2 : 119 chunks
  Cluster 3 : 64 chunks
  Cluster 4 : 191 chunks

Generating Level 1 summaries with Phi-3.5-mini...
  Summarising cluster 0 (89 chunks)...
    Summary : The Indian Military Airworthiness document (IMAP-2023) outlines a structured process for amendments to military airworthiness standards, requiring pro...
  Summarising cluster 1 (99 chunks)...
    Summary : The Indian Military Airworthiness Procedure (IMAP-2023) outlines the framework for ensuring the safety and performance of military air systems and air...
  Summarising cluster 2 (119 chunks)...
    Summary : The Indian Military Airworthiness framework involves three key entities: User Services, designe

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Meta-cluster 0 : 3 L1 summaries
  Meta-cluster 1 : 1 L1 summaries
  Meta-cluster 2 : 1 L1 summaries

Generating Level 2 meta-summaries with Phi-3.5-mini...
  Summarising meta-cluster 0...
    Meta-summary : The Indian Military Airworthiness Procedure (IMAP-2023) establishes a comprehensive framework for the airworthiness of military aircraft and airborne ...
  Summarising meta-cluster 1...
    Meta-summary : The Indian Military Airworthiness (IMAP-2023) framework establishes a tripartite structure involving User Services, designers/contractors, and Technic...
  Summarising meta-cluster 2...
    Meta-summary : The IMAP-2023 establishes a comprehensive framework for updating military airworthiness standards in India, mandating that amendments be submitted to ...

Adding RAPTOR summary nodes to Qdrant...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Summary nodes added   : 8
  Total points in Qdrant: 577

Checkpoint saved → /kaggle/working/rag_project/raptor_summaries.json

--- RAPTOR complete ---
  L1 summaries   : 5
  L2 summaries   : 3
  Total summaries: 8
  Qdrant total   : 577 points

Sample L1 summary:
  The Indian Military Airworthiness document (IMAP-2023) outlines a structured process for amendments to military airworthiness standards, requiring proposals to be submitted to the Joint Airworthiness Committee (JAC) for discussion and approval. Amendments are serially numbered and must be recorded w

Sample L2 meta-summary:
  The Indian Military Airworthiness Procedure (IMAP-2023) establishes a comprehensive framework for the airworthiness of military aircraft and airborne stores, ensuring safety and performance through a collaborative approach involving Technical Airworthiness Authorities, Main Contractors, and User Ser


In [11]:
# ============================================================
# CELL 9 — NEO4J GRAPH-RAG
# Chunk nodes + typed edges + entity extraction
# ============================================================

print("=" * 50)
print("CELL 9 — NEO4J GRAPH-RAG")
print("=" * 50 + "\n")

# ---- Part A: Connect to Neo4j ------------------------------

print("Connecting to Neo4j Aura...")
try:
    neo4j_driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD)
    )
    neo4j_driver.verify_connectivity()
    print("  Connected to Neo4j ✓")
except Exception as e:
    print(f"  Connection failed: {e}")
    raise

# ---- Part B: Clear existing graph -------------------------

print("\nClearing existing graph...")
with neo4j_driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
print("  Graph cleared ✓")

# ---- Part C: Create chunk nodes ----------------------------

print("\nCreating chunk nodes...")

def create_chunk_nodes(chunks: List[Dict]):
    with neo4j_driver.session() as session:
        for chunk in tqdm(chunks, desc="Creating nodes"):
            session.run("""
                CREATE (c:Chunk {
                    chunk_id    : $chunk_id,
                    text        : $text,
                    page        : $page,
                    section     : $section,
                    content_type: $content_type,
                    doc_name    : $doc_name,
                    char_count  : $char_count
                })
            """,
            chunk_id     = chunk["chunk_id"],
            text         = chunk["text"][:500],  # truncate for Neo4j property limit
            page         = chunk["page"],
            section      = chunk["section"],
            content_type = chunk["content_type"],
            doc_name     = chunk["doc_name"],
            char_count   = chunk["char_count"]
            )

create_chunk_nodes(all_chunks)

# Verify node count
with neo4j_driver.session() as session:
    count = session.run("MATCH (c:Chunk) RETURN count(c) as cnt").single()["cnt"]
print(f"  Chunk nodes created : {count}")

# ---- Part D: Create index on chunk_id ----------------------

print("\nCreating index on chunk_id...")
with neo4j_driver.session() as session:
    session.run("CREATE INDEX chunk_id_index IF NOT EXISTS FOR (c:Chunk) ON (c.chunk_id)")
print("  Index created ✓")

# ---- Part E: Edge type 1 — Adjacent pages ------------------

print("\nCreating ADJACENT_PAGE edges...")

# Build page lookup
page_lookup = {}
for chunk in all_chunks:
    p = chunk["page"]
    if p not in page_lookup:
        page_lookup[p] = []
    page_lookup[p].append(chunk["chunk_id"])

adjacent_edges = 0
with neo4j_driver.session() as session:
    pages = sorted(page_lookup.keys())
    for i in range(len(pages) - 1):
        p1 = pages[i]
        p2 = pages[i + 1]
        if p2 - p1 == 1:  # truly adjacent pages
            for cid1 in page_lookup[p1]:
                for cid2 in page_lookup[p2]:
                    session.run("""
                        MATCH (a:Chunk {chunk_id: $cid1})
                        MATCH (b:Chunk {chunk_id: $cid2})
                        CREATE (a)-[:ADJACENT_PAGE]->(b)
                    """, cid1=cid1, cid2=cid2)
                    adjacent_edges += 1

print(f"  ADJACENT_PAGE edges : {adjacent_edges}")

# ---- Part F: Edge type 2 — Same section --------------------

print("\nCreating SAME_SECTION edges...")

section_lookup = {}
for chunk in all_chunks:
    s = chunk["section"]
    if s not in section_lookup:
        section_lookup[s] = []
    section_lookup[s].append(chunk["chunk_id"])

same_section_edges = 0
with neo4j_driver.session() as session:
    for section, chunk_ids_in_section in section_lookup.items():
        if len(chunk_ids_in_section) < 2:
            continue
        # Connect consecutive chunks in same section only
        # Avoid O(n²) by connecting only neighbours
        for i in range(len(chunk_ids_in_section) - 1):
            cid1 = chunk_ids_in_section[i]
            cid2 = chunk_ids_in_section[i + 1]
            session.run("""
                MATCH (a:Chunk {chunk_id: $cid1})
                MATCH (b:Chunk {chunk_id: $cid2})
                CREATE (a)-[:SAME_SECTION {section: $section}]->(b)
            """, cid1=cid1, cid2=cid2, section=section)
            same_section_edges += 1

print(f"  SAME_SECTION edges  : {same_section_edges}")

# ---- Part G: Edge type 3 — High semantic similarity --------

print("\nCreating HIGH_SIMILARITY edges...")

SIMILARITY_EDGE_THRESHOLD = 0.85
similarity_edges = 0

# Compare all text chunk pairs — use batched dot product
text_chunk_ids   = [c["chunk_id"] for c in all_chunks if c["content_type"] == "text"]
text_chunk_embs  = np.array([
    embedding_store[cid]["dense_vector"]
    for cid in text_chunk_ids
    if cid in embedding_store
])

# Compute similarity matrix in blocks to avoid memory issues
BLOCK_SIZE = 100
with neo4j_driver.session() as session:
    for i in range(0, len(text_chunk_ids), BLOCK_SIZE):
        block_embs = text_chunk_embs[i:i+BLOCK_SIZE]
        sims       = np.dot(block_embs, text_chunk_embs.T)

        for local_idx, global_sims in enumerate(sims):
            global_idx = i + local_idx
            # Find chunks with similarity above threshold
            # Exclude self and already-covered pairs
            for j in range(global_idx + 1, len(text_chunk_ids)):
                if global_sims[j] >= SIMILARITY_EDGE_THRESHOLD:
                    cid1 = text_chunk_ids[global_idx]
                    cid2 = text_chunk_ids[j]
                    session.run("""
                        MATCH (a:Chunk {chunk_id: $cid1})
                        MATCH (b:Chunk {chunk_id: $cid2})
                        CREATE (a)-[:HIGH_SIMILARITY {score: $score}]->(b)
                    """, cid1=cid1, cid2=cid2,
                         score=round(float(global_sims[j]), 4))
                    similarity_edges += 1

print(f"  HIGH_SIMILARITY edges : {similarity_edges}")

# ---- Part H: Edge type 4 — Shared entities -----------------

print("\nExtracting entities and creating SHARES_ENTITY edges...")
print("  (This takes a few minutes — Phi-3.5-mini per chunk sample)")

# Sample chunks for entity extraction — every 5th text chunk
# Full extraction would take too long with 562 chunks
sample_chunks = [
    c for i, c in enumerate(all_chunks)
    if c["content_type"] == "text" and i % 5 == 0
]
print(f"  Sampling {len(sample_chunks)} chunks for entity extraction...")

chunk_entities = {}

for chunk in tqdm(sample_chunks, desc="Extracting entities"):
    prompt = f"""Extract named entities from this aviation regulatory text.
Return ONLY a comma-separated list of entity names. No explanation.
Entity types to find: regulatory bodies, document names, procedures, certification types, roles.

Text: {chunk['text'][:400]}

Entities:"""

    response = run_phi3(prompt, max_new_tokens=80)
    # Parse comma-separated entities
    entities = [e.strip().lower() for e in response.split(",") if len(e.strip()) > 2]
    chunk_entities[chunk["chunk_id"]] = entities

# Find chunk pairs that share entities and create edges
entity_edges = 0
chunk_id_list = list(chunk_entities.keys())

with neo4j_driver.session() as session:
    for i in range(len(chunk_id_list)):
        for j in range(i + 1, len(chunk_id_list)):
            cid1 = chunk_id_list[i]
            cid2 = chunk_id_list[j]
            shared = set(chunk_entities[cid1]) & set(chunk_entities[cid2])
            if shared:
                shared_str = ", ".join(list(shared)[:5])
                session.run("""
                    MATCH (a:Chunk {chunk_id: $cid1})
                    MATCH (b:Chunk {chunk_id: $cid2})
                    CREATE (a)-[:SHARES_ENTITY {entities: $entities}]->(b)
                """, cid1=cid1, cid2=cid2, entities=shared_str)
                entity_edges += 1

print(f"  SHARES_ENTITY edges : {entity_edges}")

# ---- Part I: Graph summary ---------------------------------

print("\nQuerying graph statistics...")
with neo4j_driver.session() as session:
    node_count = session.run(
        "MATCH (n) RETURN count(n) as cnt"
    ).single()["cnt"]

    edge_count = session.run(
        "MATCH ()-[r]->() RETURN count(r) as cnt"
    ).single()["cnt"]

    edge_types = session.run("""
        MATCH ()-[r]->()
        RETURN type(r) as edge_type, count(r) as cnt
        ORDER BY cnt DESC
    """).data()

print(f"\n--- Neo4j graph-RAG complete ---")
print(f"  Nodes : {node_count}")
print(f"  Edges : {edge_count}")
print(f"\n  Edge breakdown:")
for et in edge_types:
    print(f"    {et['edge_type']:<20} : {et['cnt']}")

# ---- Part J: Graph traversal helper ------------------------

def graph_retrieve(seed_chunk_ids: List[str], top_k: int = TOP_K_GRAPH) -> List[Dict]:
    """
    Given seed chunk_ids from vector search,
    traverse Neo4j to find connected neighbour chunks.
    Returns list of neighbour chunk dicts.
    """
    neighbours = []
    seen       = set(seed_chunk_ids)

    with neo4j_driver.session() as session:
        for seed_id in seed_chunk_ids:
            result = session.run("""
                MATCH (seed:Chunk {chunk_id: $chunk_id})
                      -[r:ADJACENT_PAGE|SAME_SECTION|HIGH_SIMILARITY|SHARES_ENTITY]-
                      (neighbour:Chunk)
                WHERE neighbour.chunk_id <> $chunk_id
                RETURN DISTINCT
                    neighbour.chunk_id    AS chunk_id,
                    neighbour.text        AS text,
                    neighbour.page        AS page,
                    neighbour.section     AS section,
                    neighbour.content_type AS content_type,
                    neighbour.doc_name    AS doc_name,
                    type(r)               AS edge_type
                LIMIT $limit
            """, chunk_id=seed_id, limit=top_k)

            for record in result:
                cid = record["chunk_id"]
                if cid not in seen:
                    seen.add(cid)
                    neighbours.append({
                        "chunk_id"    : cid,
                        "text"        : record["text"],
                        "page"        : record["page"],
                        "section"     : record["section"],
                        "content_type": record["content_type"],
                        "doc_name"    : record["doc_name"],
                        "edge_type"   : record["edge_type"]
                    })

    return neighbours[:top_k]


# ---- Part K: Quick traversal test -------------------------

print("\nRunning graph traversal test...")
# Use first text chunk as seed
seed = next(c for c in all_chunks if c["content_type"] == "text")
neighbours = graph_retrieve([seed["chunk_id"]], top_k=3)

print(f"  Seed chunk  : page {seed['page']} — {seed['text'][:80]}")
print(f"  Neighbours found : {len(neighbours)}")
for n in neighbours:
    print(f"\n  → [{n['edge_type']}] page {n['page']}")
    print(f"    {n['text'][:100]}")

print("\n--- Cell 9 complete ---")

CELL 9 — NEO4J GRAPH-RAG

Connecting to Neo4j Aura...
  Connected to Neo4j ✓

Clearing existing graph...
  Graph cleared ✓

Creating chunk nodes...


Creating nodes: 100%|██████████| 569/569 [01:38<00:00,  5.77it/s]


  Chunk nodes created : 569

Creating index on chunk_id...
  Index created ✓

Creating ADJACENT_PAGE edges...
  ADJACENT_PAGE edges : 3003

Creating SAME_SECTION edges...
  SAME_SECTION edges  : 479

Creating HIGH_SIMILARITY edges...
  HIGH_SIMILARITY edges : 357

Extracting entities and creating SHARES_ENTITY edges...
  (This takes a few minutes — Phi-3.5-mini per chunk sample)
  Sampling 113 chunks for entity extraction...


Extracting entities: 100%|██████████| 113/113 [04:48<00:00,  2.56s/it]


  SHARES_ENTITY edges : 1464

Querying graph statistics...

--- Neo4j graph-RAG complete ---
  Nodes : 569
  Edges : 5303

  Edge breakdown:
    ADJACENT_PAGE        : 3003
    SHARES_ENTITY        : 1464
    SAME_SECTION         : 479
    HIGH_SIMILARITY      : 357

Running graph traversal test...
  Seed chunk  : page 1 — INDIAN MILITARY 
AIRWORTHINESS
PROCEDURE - 2023
IMAP-2023 MINISTRY OF DEFENCE
GO
  Neighbours found : 2

  → [ADJACENT_PAGE] page 2
    Indian Military 
Airworthiness Procedure - 2023 
(IMAP-2023) Suggestions for improvement of this doc

  → [ADJACENT_PAGE] page 2
    DRDO, Ministry of Defence (R & D)

--- Cell 9 complete ---


In [12]:
# ============================================================
# CELL 10 — HYBRID RETRIEVAL
# Dense + Sparse + Graph → RRF → MMR → Reranking
# ============================================================

print("=" * 50)
print("CELL 10 — HYBRID RETRIEVAL")
print("=" * 50 + "\n")

# ---- Part A: Neo4j driver ----------------------------------

neo4j_driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD)
)

def get_graph_neighbours(seed_chunk_ids: List[str], limit: int = 10) -> List[Dict]:
    """
    Given seed chunk_ids from vector search,
    traverse Neo4j graph to find connected neighbours.
    Returns list of neighbour chunk dicts.
    """
    with neo4j_driver.session() as session:
        result = session.run("""
            MATCH (seed:Chunk)
            WHERE seed.chunk_id IN $chunk_ids
            MATCH (seed)-[r:ADJACENT_PAGE|SAME_SECTION|HIGH_SIMILARITY|SHARES_ENTITY]-(neighbour:Chunk)
            WHERE NOT neighbour.chunk_id IN $chunk_ids
            RETURN DISTINCT
                neighbour.chunk_id  AS chunk_id,
                neighbour.text      AS text,
                neighbour.page      AS page,
                neighbour.section   AS section,
                neighbour.doc_name  AS doc_name,
                type(r)             AS edge_type,
                count(r)            AS connection_count
            ORDER BY connection_count DESC
            LIMIT $limit
        """, chunk_ids=seed_chunk_ids, limit=limit)

        neighbours = []
        for record in result:
            neighbours.append({
                "chunk_id"    : record["chunk_id"],
                "text"        : record["text"],
                "page"        : record["page"],
                "section"     : record["section"],
                "doc_name"    : record["doc_name"],
                "content_type": "text",
                "edge_type"   : record["edge_type"],
                "char_count"  : len(record["text"]) if record["text"] else 0
            })
        return neighbours


# ---- Part B: Dense retrieval -------------------------------

def dense_search(query_embedding: List[float], top_k: int = TOP_K_DENSE) -> List[Dict]:
    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding,
        using="dense",
        limit=top_k
    ).points

    chunks = []
    for r in results:
        chunks.append({
            "chunk_id"    : r.payload["chunk_id"],
            "text"        : r.payload["text"],
            "page"        : r.payload["page"],
            "section"     : r.payload["section"],
            "doc_name"    : r.payload["doc_name"],
            "content_type": r.payload["content_type"],
            "char_count"  : r.payload["char_count"],
            "score"       : r.score
        })
    return chunks


# ---- Part C: Sparse retrieval ------------------------------

def sparse_search(query_text: str, top_k: int = TOP_K_SPARSE) -> List[Dict]:
    # Tokenize query
    query_tokens = re.findall(r'\b[a-zA-Z]{2,}\b', query_text.lower())

    if not query_tokens:
        return []

    # Compute BM25 query vector
    query_indices = []
    query_values  = []
    tf_counts     = {}
    for token in query_tokens:
        tf_counts[token] = tf_counts.get(token, 0) + 1

    for token, tf in tf_counts.items():
        if token in bm25.idf:
            query_indices.append(abs(hash(token)) % (10**6))
            query_values.append(float(tf * bm25.idf[token]))

    if not query_indices:
        return []

    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=SparseVector(
            indices=query_indices,
            values=query_values
        ),
        using="sparse",
        limit=top_k
    ).points

    chunks = []
    for r in results:
        chunks.append({
            "chunk_id"    : r.payload["chunk_id"],
            "text"        : r.payload["text"],
            "page"        : r.payload["page"],
            "section"     : r.payload["section"],
            "doc_name"    : r.payload["doc_name"],
            "content_type": r.payload["content_type"],
            "char_count"  : r.payload["char_count"],
            "score"       : r.score
        })
    return chunks


# ---- Part D: RRF Fusion ------------------------------------

def rrf_fusion(result_lists: List[List[Dict]], k: int = 60) -> List[Dict]:
    """
    Reciprocal Rank Fusion across multiple result lists.
    score = sum(1 / (rank + k)) across all lists.
    """
    scores   = {}
    chunk_map = {}

    for result_list in result_lists:
        for rank, chunk in enumerate(result_list):
            cid = chunk["chunk_id"]
            if cid not in scores:
                scores[cid]    = 0.0
                chunk_map[cid] = chunk
            scores[cid] += 1.0 / (rank + k)

    # Sort by RRF score descending
    sorted_ids = sorted(scores.keys(), key=lambda x: scores[x], reverse=True)
    fused = []
    for cid in sorted_ids:
        chunk = chunk_map[cid].copy()
        chunk["rrf_score"] = scores[cid]
        fused.append(chunk)

    return fused


# ---- Part E: MMR Diversity ---------------------------------

def mmr_selection(
    query_embedding : np.ndarray,
    candidates      : List[Dict],
    top_k           : int,
    lambda_param    : float = MMR_LAMBDA
) -> List[Dict]:
    """
    Maximal Marginal Relevance selection.
    Uses precomputed embeddings from embedding_store — no re-encoding.
    lambda_param controls relevance vs diversity tradeoff.
    """
    if not candidates:
        return []

    # Get precomputed embeddings for candidates
    candidate_embeddings = []
    valid_candidates     = []

    for chunk in candidates:
        cid = chunk["chunk_id"]
        if cid in embedding_store:
            emb = np.array(embedding_store[cid]["dense_vector"])
            candidate_embeddings.append(emb)
            valid_candidates.append(chunk)

    if not valid_candidates:
        return candidates[:top_k]

    candidate_embeddings = np.array(candidate_embeddings)
    selected_indices     = []
    selected_embeddings  = []

    for _ in range(min(top_k, len(valid_candidates))):
        if not selected_indices:
            # First selection — pick highest RRF score
            scores = [c.get("rrf_score", 0) for c in valid_candidates]
            best   = int(np.argmax(scores))
        else:
            # MMR score = lambda * relevance - (1-lambda) * max_similarity_to_selected
            relevance = np.dot(candidate_embeddings, query_embedding)

            selected_mat = np.array(selected_embeddings)
            similarity_to_selected = np.max(
                np.dot(candidate_embeddings, selected_mat.T), axis=1
            )

            mmr_scores = (
                lambda_param * relevance -
                (1 - lambda_param) * similarity_to_selected
            )

            # Mask already selected
            for idx in selected_indices:
                mmr_scores[idx] = -np.inf

            best = int(np.argmax(mmr_scores))

        selected_indices.append(best)
        selected_embeddings.append(candidate_embeddings[best])

    return [valid_candidates[i] for i in selected_indices]


# ---- Part F: Full retrieval pipeline -----------------------

def retrieve(query: str, top_k: int = TOP_K_FINAL) -> List[Dict]:
    """
    Full hybrid retrieval pipeline:
    Dense + Sparse + Graph → RRF → MMR → Reranking
    """
    print(f"\n  Query: '{query}'")

    # 1. Embed query
    query_embedding = get_embeddings(
        ["Represent this sentence for searching relevant passages: " + query],
        batch_size=1
    )[0]

    # 2. Dense search
    dense_results = dense_search(query_embedding.tolist(), TOP_K_DENSE)
    print(f"  Dense results  : {len(dense_results)}")

    # 3. Sparse search
    sparse_results = sparse_search(query, TOP_K_SPARSE)
    print(f"  Sparse results : {len(sparse_results)}")

    # 4. Graph traversal — use top 5 dense seeds
    seed_ids      = [c["chunk_id"] for c in dense_results[:5]]
    graph_results = get_graph_neighbours(seed_ids, TOP_K_GRAPH)
    print(f"  Graph results  : {len(graph_results)}")

    # 5. RRF fusion of all three lists
    fused = rrf_fusion([dense_results, sparse_results, graph_results])
    print(f"  After RRF      : {len(fused)}")

    # 6. MMR diversity selection
    mmr_results = mmr_selection(
        query_embedding=query_embedding,
        candidates=fused,
        top_k=top_k * 3,
        lambda_param=MMR_LAMBDA
    )
    print(f"  After MMR      : {len(mmr_results)}")

    # 7. Cross-encoder reranking
    pairs  = [(query, c["text"]) for c in mmr_results]
    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(mmr_results, scores),
        key=lambda x: x[1],
        reverse=True
    )
    final = [chunk for chunk, score in ranked[:top_k]]

    print(f"  After rerank   : {len(final)}")
    return final


# ---- Part G: Test retrieval --------------------------------

print("Testing hybrid retrieval pipeline...\n")

test_queries = [
    "What is the role of CEMILAC in airworthiness certification?",
    "What are the flight testing requirements for military aircraft?",
    "How are amendments to IMAP-2023 approved?"
]

for query in test_queries:
    print("-" * 50)
    results = retrieve(query)
    print(f"\n  Top {len(results)} results:")
    for i, chunk in enumerate(results):
        print(f"\n  [{i+1}] page    : {chunk['page']}")
        print(f"       section : {chunk['section'][:60]}")
        print(f"       type    : {chunk['content_type']}")
        print(f"       text    : {chunk['text'][:200]}")

print("\n--- Cell 10 complete ---")

CELL 10 — HYBRID RETRIEVAL

Testing hybrid retrieval pipeline...

--------------------------------------------------

  Query: 'What is the role of CEMILAC in airworthiness certification?'


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Dense results  : 20
  Sparse results : 20
  Graph results  : 10
  After RRF      : 43
  After MMR      : 18
  After rerank   : 6

  Top 6 results:

  [1] page    : 18
       section : 2.3	 Technical Airworthiness Authorities
       type    : text
       text    : Centre for Military Airworthiness and Certification (CEMILAC), under the 
Dept. of Defence Research and Development, is the military airworthiness 
certification authority responsible for grant of ini

  [2] page    : 17
       section : General
       type    : text
       text    : a.	CEMILAC – Regulatory Authority for 
Design Approval b.	DGAQA – Regulatory Authority for 
Quality Assurance Approval a. Carry out technical airworthiness activities 
during design, development and p

  [3] page    : 118
       section : 12.5.2  Clearance Procedure for Non-Critical Airborne Stores
       type    : text
       text    : System Certification Review Board (SCRB)
SCRB is a board constituted by CE(A), CEMILAC, if required, to addres

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Dense results  : 20
  Sparse results : 20
  Graph results  : 10
  After RRF      : 42
  After MMR      : 18
  After rerank   : 6

  Top 6 results:

  [1] page    : 70
       section : 5.2	 Flight Testing of Ab-initio Developed Aircraft & UAS
       type    : text
       text    : Flight Testing of Air Systems and Airborne Stores
Part 2 - Chapter 5 a. Flight testing shall be carried out on an Air System registered under Indian Military 
Register with the User Services or which 

  [2] page    : 112
       section : 12.5.2  Clearance Procedure for Non-Critical Airborne Stores
       type    : text
       text    : Flight Test Plan
A flight test plan typically defines the flight testing requirements for a particular phase of 
flight, including objectives, Air System, trial dates, venue, pre-requisites, SOP, conf

  [3] page    : 112
       section : 12.5.2  Clearance Procedure for Non-Critical Airborne Stores
       type    : text
       text    : Flight Test Specification
Flight test s

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Dense results  : 20
  Sparse results : 20
  Graph results  : 10
  After RRF      : 31
  After MMR      : 18
  After rerank   : 6

  Top 6 results:

  [1] page    : 19
       section : 2.4	 Indian Military Airworthiness Framework
       type    : text
       text    : The airworthiness procedure document called the IMAP-2023 is the apex governing 
document for Indian Military Airworthiness. The document is drafted by CEMILAC and 
reviewed by Joint Airworthiness Com

  [2] page    : 14
       section : General
       type    : text
       text    : This document shall be reviewed by the advisory body every 3 years for possible 
updates keeping in mind the contemporary advancements in military aviation and the 
suggestions received. This document

  [3] page    : 13
       section : General
       type    : text
       text    : This document titled, Indian Military Airworthiness Procedure-2023 (IMAP-2023), 
is a Procedural document on Technical Airworthiness, covering roles and respons

In [14]:
# ============================================================
# CELL 11 — HyDE + GENERATION
# Hypothetical Document Embedding fused with standard RAG
# Grounded answer generation with page citations
# ============================================================

print("=" * 50)
print("CELL 11 — HyDE + GENERATION")
print("=" * 50 + "\n")

# ---- Part A: HyDE retrieval --------------------------------

def hyde_retrieve(query: str, top_k: int = TOP_K_FINAL) -> List[Dict]:
    """
    HyDE — Generate a hypothetical answer, embed it,
    fuse with standard dense retrieval via RRF.
    Then run full hybrid pipeline on fused results.
    """

    # Step 1 — Generate hypothetical document
    hyde_prompt = f"""You are an expert on Indian Military Airworthiness procedures.
A user asked: "{query}"

Write a short hypothetical passage (3-5 sentences) that would perfectly answer this question,
using terminology from Indian military aviation and airworthiness regulations.
Write only the passage, no preamble."""

    hypothetical_doc = run_phi3(hyde_prompt, max_new_tokens=150)

    # Step 2 — Embed both query and hypothetical doc
    query_embedding = get_embeddings(
        ["Represent this sentence for searching relevant passages: " + query],
        batch_size=1
    )[0]

    hyde_embedding = get_embeddings(
        [hypothetical_doc],
        batch_size=1
    )[0]

    # Step 3 — Dense search with both embeddings
    standard_dense = dense_search(query_embedding.tolist(), TOP_K_DENSE)
    hyde_dense     = dense_search(hyde_embedding.tolist(), TOP_K_DENSE)

    # Step 4 — Sparse search
    sparse_results = sparse_search(query, TOP_K_SPARSE)

    # Step 5 — Graph traversal using standard dense seeds
    seed_ids      = [c["chunk_id"] for c in standard_dense[:5]]
    graph_results = get_graph_neighbours(seed_ids, TOP_K_GRAPH)

    # Step 6 — RRF fusion of all four lists
    fused = rrf_fusion([standard_dense, hyde_dense, sparse_results, graph_results])

    # Step 7 — MMR diversity
    mmr_results = mmr_selection(
        query_embedding=query_embedding,
        candidates=fused,
        top_k=top_k * 3,
        lambda_param=MMR_LAMBDA
    )

    # Step 8 — Reranking
    pairs  = [(query, c["text"]) for c in mmr_results]
    scores = reranker.predict(pairs)
    ranked = sorted(
        zip(mmr_results, scores),
        key=lambda x: x[1],
        reverse=True
    )
    final = [chunk for chunk, score in ranked[:top_k]]

    return final, hypothetical_doc


# ---- Part B: Grounded answer generation --------------------

def generate_answer(query: str, context_chunks: List[Dict]) -> str:
    """
    Generate a grounded answer using Phi-3.5-mini.
    Strict no-hallucination instruction.
    Full chunk text passed — not truncated snippets.
    Page citations included.
    """
    # Build context string with page citations
    context_parts = []
    for i, chunk in enumerate(context_chunks):
        context_parts.append(
            f"[Source {i+1} — Page {chunk['page']}, {chunk['section']}]\n{chunk['text']}"
        )
    context = "\n\n".join(context_parts)

    prompt = f"""You are an expert assistant on Indian Military Airworthiness Procedures (IMAP-2023).
Answer the question using ONLY the provided source passages.
Cite page numbers for every claim you make using (Page X) format.
If the answer is not found in the sources, say "Not found in the provided context."
Do not add information from outside the sources.

Question: {query}

Sources:
{context}

Answer:"""

    answer = run_phi3(prompt, max_new_tokens=512)
    return answer


# ---- Part C: Full RAG pipeline -----------------------------

def rag_pipeline(query: str) -> Dict:
    """
    Full pipeline — HyDE retrieval + grounded generation.
    Returns query, hypothetical doc, retrieved chunks, answer.
    """
    print(f"\nQuery: {query}")
    print("-" * 50)

    # Retrieve with HyDE
    chunks, hypothetical_doc = hyde_retrieve(query)

    print(f"Hypothetical doc : {hypothetical_doc[:150]}...")
    print(f"Retrieved chunks : {len(chunks)}")

    # Generate answer
    print("Generating answer...")
    answer = generate_answer(query, chunks)

    print(f"\nAnswer:\n{answer}")

    return {
        "query"          : query,
        "hypothetical_doc": hypothetical_doc,
        "chunks"         : chunks,
        "answer"         : answer
    }


# ---- Part D: Test pipeline ---------------------------------

print("Testing HyDE + Generation pipeline...\n")

test_queries = [
    "What is the role of CEMILAC in airworthiness certification?",
    "What are the flight testing requirements for military aircraft?",
    "How are amendments to IMAP-2023 approved?"
]

results = []
for query in test_queries:
    print("=" * 60)
    result = rag_pipeline(query)
    results.append(result)
    print()

# ---- Part E: Save results ----------------------------------

results_path = f"{PROJECT_DIR}/generation_results.json"

# Convert chunks to serializable format
serializable = []
for r in results:
    serializable.append({
        "query"           : r["query"],
        "hypothetical_doc": r["hypothetical_doc"],
        "answer"          : r["answer"],
        "chunks"          : [
            {k: v for k, v in c.items() if k != "score"}
            for c in r["chunks"]
        ]
    })

with open(results_path, "w") as f:
    json.dump(serializable, f, indent=2)

print(f"\nResults saved → {results_path}")
print("\n--- Cell 11 complete ---")

CELL 11 — HyDE + GENERATION

Testing HyDE + Generation pipeline...


Query: What is the role of CEMILAC in airworthiness certification?
--------------------------------------------------


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Hypothetical doc : CEMILAC, the Central Military Institute of Aviation Civil Aircraft (Civil Aircraft) in India, plays a pivotal role in airworthiness certification by e...
Retrieved chunks : 6
Generating answer...

Answer:
CEMILAC plays a crucial role in airworthiness certification as the military airworthiness certification authority responsible for granting initial airworthiness approvals and continued airworthiness approvals (Source 1 — Page 18, 2.3). It carries out these activities through its Regional Centres for Military Airworthiness (RCMAs) (Source 1 — Page 18, 2.3). Additionally, CEMILAC may approve the setting up of Airworthiness Groups in approved design organizations for the progression of airworthiness certification on behalf of CEMILAC (Source 4 — Page 31, 1.4.1). The organization also approves Airworthiness Certification Plans (ACPs) to ensure compliance with Technical Compliance Board (TCB) requirements for Air Systems, involving various stakeholders at different stage

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Hypothetical doc : In accordance with Indian Military Airworthiness regulations, flight testing for military aircraft encompasses rigorous evaluation of structural integ...
Retrieved chunks : 6
Generating answer...

Answer:
According to the provided sources, the flight testing requirements for military aircraft are outlined as follows:

1. Flight testing shall be carried out on an Air System registered under the Indian Military Register with the User Services or which has been issued with a military tail number (Source 1 — Page 70, 5.2).

2. A flight test plan typically defines the flight testing requirements for a particular phase of flight, including objectives, Air System, trial dates, venue, pre-requisites, SOP, configuration, tests to be conducted, environment, support, and instrumentation needs (Source 2 — Page 112, 12.5.2).

3. Flight test specifications are the demonstration requirements for an Air System/Airborne Stores that need to be verified through flight tests towards com

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Hypothetical doc : Amendments to IMAP-2023 are approved through a rigorous process involving the Defence Airworthiness Authority (DAA), which evaluates the proposed chan...
Retrieved chunks : 6
Generating answer...

Answer:
Amendments to IMAP-2023 are approved through a structured process as outlined in the provided sources. According to Source 4 (Page 2), suggestions for improvement of IMAP-2023 should be addressed to the Joint Airworthiness Committee (JAC) Centre for Military Airworthiness & Certification. This implies that the JAC Centre is responsible for reviewing and considering amendments to the document.

Furthermore, Source 2 (Page 14) states that the document shall be reviewed by the advisory body every 3 years for possible updates, taking into account contemporary advancements in military aviation and suggestions received. This indicates that the process for approving amendments involves periodic reviews and consultations with relevant stakeholders.

In summary, amendments t

In [15]:
# ============================================================
# CELL 12 — EVALUATION
# Faithfulness, Answer Relevancy, Context Precision,
# Groundedness — fixed NLI truncation
# ============================================================

print("=" * 50)
print("CELL 12 — EVALUATION")
print("=" * 50 + "\n")

# ---- Part A: 10 test questions with reference answers ------
# Reference answers verified against actual PDF content

test_cases = [
    {
        "question": "What is CEMILAC and what is its primary role?",
        "reference": "CEMILAC is the Centre for Military Airworthiness and Certification under the Department of Defence Research and Development. It is the military airworthiness certification authority responsible for grant of initial and continued airworthiness approvals."
    },
    {
        "question": "What does DGAQA stand for and what is its role?",
        "reference": "DGAQA stands for Directorate General of Aeronautical Quality Assurance. It is the regulatory authority for quality assurance approval for Indian military air systems and airborne stores."
    },
    {
        "question": "What is the Joint Airworthiness Committee (JAC)?",
        "reference": "The Joint Airworthiness Committee is the advisory body that reviews IMAP-2023 every 3 years and receives suggestions for improvement. It oversees the Indian military airworthiness framework."
    },
    {
        "question": "What is an Airworthiness Certification Plan?",
        "reference": "An Airworthiness Certification Plan is prepared by the Main Contractor bringing out the design and development, test and evaluation details towards compliance with Technical Compliance Board requirements for Air Systems."
    },
    {
        "question": "How often is IMAP-2023 reviewed for updates?",
        "reference": "IMAP-2023 is reviewed by the advisory body every 3 years for possible updates keeping in mind contemporary advancements in military aviation and suggestions received."
    },
    {
        "question": "What are the requirements for flight testing personnel?",
        "reference": "Flight test crew, both the Test Pilot and the Flight Test Engineer, shall be a graduate of a recognized Test Pilot School or shall have undergone a suitable course on flight testing."
    },
    {
        "question": "What is the Indian Military Register?",
        "reference": "The Indian Military Register is the register under which Air Systems used for flight testing must be registered with User Services or issued a military tail number."
    },
    {
        "question": "What is a Design Organisation Approval?",
        "reference": "Design Organisation Approval is an approval granted by CEMILAC to design organisations for progression of airworthiness certification activities on behalf of CEMILAC."
    },
    {
        "question": "What is the System Certification Review Board?",
        "reference": "The System Certification Review Board is a board constituted by CE(A) and CEMILAC to address issues related to airworthiness certification, chaired by CE(A) or a CEMILAC representative."
    },
    {
        "question": "What is the role of the Main Contractor in airworthiness?",
        "reference": "The Main Contractor is responsible for ensuring the Air System is designed to applicable airworthiness certification criteria and for preparing the Airworthiness Certification Plan in consultation with CEMILAC."
    }
]

# ---- Part B: NLI helper — fixed truncation -----------------

def compute_nli_score(premise: str, hypothesis: str) -> float:
    """
    Compute entailment probability between premise and hypothesis.
    Fixed truncation — truncate each separately before concatenating.
    """
    # Truncate each to 200 chars independently
    premise_trunc    = premise[:200]
    hypothesis_trunc = hypothesis[:200]

    try:
        result = nli_pipeline(
            f"{premise_trunc} [SEP] {hypothesis_trunc}",
            truncation=True,
            max_length=512
        )
        # Find entailment score
        for r in result:
            if isinstance(r, list):
                for item in r:
                    if "entail" in item["label"].lower():
                        return item["score"]
            else:
                if "entail" in r["label"].lower():
                    return r["score"]
        return 0.0
    except Exception as e:
        return 0.0


# ---- Part C: Evaluation metrics ----------------------------

def evaluate_answer(
    question   : str,
    answer     : str,
    chunks     : List[Dict],
    reference  : str
) -> Dict:
    """
    Compute four metrics:
    1. Faithfulness    — NLI: does context entail the answer?
    2. Answer Relevancy — BGE cosine: answer vs question
    3. Context Precision — BGE cosine: chunks vs question
    4. Groundedness    — NLI per sentence: answer sentences entailed by context
    """
    context_text = " ".join([c["text"] for c in chunks])

    # 1. Faithfulness — NLI(context, answer)
    faithfulness = compute_nli_score(context_text, answer)

    # 2. Answer Relevancy — cosine(answer_emb, question_emb)
    embs = get_embeddings([answer, question], batch_size=2)
    answer_relevancy = float(np.dot(embs[0], embs[1]))

    # 3. Context Precision — avg cosine(chunk, question) for top chunks
    question_emb = get_embeddings([question], batch_size=1)[0]
    chunk_scores = []
    for chunk in chunks:
        chunk_emb = np.array(embedding_store[chunk["chunk_id"]]["dense_vector"])
        score     = float(np.dot(chunk_emb, question_emb))
        chunk_scores.append(score)
    context_precision = float(np.mean(chunk_scores)) if chunk_scores else 0.0

    # 4. Groundedness — per sentence NLI
    answer_sentences = sent_tokenize(answer)
    sentence_scores  = []
    for sent in answer_sentences:
        if len(sent.strip()) < 10:
            continue
        score = compute_nli_score(context_text, sent)
        sentence_scores.append(score)
    groundedness = float(np.mean(sentence_scores)) if sentence_scores else 0.0

    return {
        "faithfulness"     : round(faithfulness, 4),
        "answer_relevancy" : round(answer_relevancy, 4),
        "context_precision": round(context_precision, 4),
        "groundedness"     : round(groundedness, 4)
    }


# ---- Part D: Run evaluation --------------------------------

print("Running evaluation on 10 test questions...\n")
all_results = []

for i, tc in enumerate(test_cases):
    print(f"[{i+1}/10] {tc['question'][:60]}...")

    # Get answer from pipeline
    chunks, hypothetical_doc = hyde_retrieve(tc["question"])
    answer = generate_answer(tc["question"], chunks)

    # Evaluate
    metrics = evaluate_answer(
        question  = tc["question"],
        answer    = answer,
        chunks    = chunks,
        reference = tc["reference"]
    )

    all_results.append({
        "question"         : tc["question"],
        "reference"        : tc["reference"],
        "answer"           : answer,
        "hypothetical_doc" : hypothetical_doc,
        "metrics"          : metrics
    })

    print(f"  Faithfulness     : {metrics['faithfulness']:.4f}")
    print(f"  Answer Relevancy : {metrics['answer_relevancy']:.4f}")
    print(f"  Context Precision: {metrics['context_precision']:.4f}")
    print(f"  Groundedness     : {metrics['groundedness']:.4f}")
    print()

# ---- Part E: Aggregate scores ------------------------------

print("=" * 50)
print("AGGREGATE EVALUATION SCORES")
print("=" * 50)

metrics_keys = ["faithfulness", "answer_relevancy", "context_precision", "groundedness"]
for key in metrics_keys:
    scores = [r["metrics"][key] for r in all_results]
    print(f"  {key:<20} : {np.mean(scores):.4f} (min {min(scores):.4f} / max {max(scores):.4f})")

# ---- Part F: Save evaluation results -----------------------

eval_path = f"{PROJECT_DIR}/evaluation_results.json"
with open(eval_path, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nEvaluation results saved → {eval_path}")
print("\n--- Cell 12 complete ---")

CELL 12 — EVALUATION

Running evaluation on 10 test questions...

[1/10] What is CEMILAC and what is its primary role?...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Faithfulness     : 0.0000
  Answer Relevancy : 0.9064
  Context Precision: 0.7850
  Groundedness     : 0.0000

AGGREGATE EVALUATION SCORES
  faithfulness         : 0.2503 (min 0.0000 / max 0.9848)
  answer_relevancy     : 0.8496 (min 0.7754 / max 0.9064)
  context_precision    : 0.7383 (min 0.6817 / max 0.7941)
  groundedness         : 0.0226 (min 0.0000 / max 0.1630)

Evaluation results saved → /kaggle/working/rag_project/evaluation_results.json

--- Cell 12 complete ---


In [18]:
def compute_nli_score(premise: str, hypothesis: str) -> float:
    premise_trunc    = premise[:300]
    hypothesis_trunc = hypothesis[:200]
    try:
        result = nli_pipeline(
            f"{premise_trunc} [SEP] {hypothesis_trunc}",
            truncation=True,
            max_length=512
        )
        for r in result:
            if "entail" in r["label"].lower():
                return r["score"]
        return 0.0
    except:
        return 0.0


def evaluate_answer(question, answer, chunks, reference) -> Dict:
    # For faithfulness — test answer against each chunk separately, take max
    faithfulness_scores = []
    for chunk in chunks:
        score = compute_nli_score(chunk["text"], answer)
        faithfulness_scores.append(score)
    faithfulness = float(max(faithfulness_scores)) if faithfulness_scores else 0.0

    # Answer relevancy
    embs = get_embeddings([answer, question], batch_size=2)
    answer_relevancy = float(np.dot(embs[0], embs[1]))

    # Context precision
    question_emb = get_embeddings([question], batch_size=1)[0]
    chunk_scores = []
    for chunk in chunks:
        chunk_emb = np.array(embedding_store[chunk["chunk_id"]]["dense_vector"])
        chunk_scores.append(float(np.dot(chunk_emb, question_emb)))
    context_precision = float(np.mean(chunk_scores)) if chunk_scores else 0.0

    # Groundedness — per sentence, test against each chunk, take max per sentence
    answer_sentences = sent_tokenize(answer)
    sentence_scores  = []
    for sent in answer_sentences:
        if len(sent.strip()) < 10:
            continue
        sent_chunk_scores = [compute_nli_score(chunk["text"], sent) for chunk in chunks]
        sentence_scores.append(max(sent_chunk_scores))
    groundedness = float(np.mean(sentence_scores)) if sentence_scores else 0.0

    return {
        "faithfulness"     : round(faithfulness, 4),
        "answer_relevancy" : round(answer_relevancy, 4),
        "context_precision": round(context_precision, 4),
        "groundedness"     : round(groundedness, 4)
    }

In [19]:
print("Re-running evaluation with fixed NLI...\n")
all_results = []

for i, tc in enumerate(test_cases):
    print(f"[{i+1}/10] {tc['question'][:60]}...")
    chunks, hypothetical_doc = hyde_retrieve(tc["question"])
    answer = generate_answer(tc["question"], chunks)
    metrics = evaluate_answer(
        question  = tc["question"],
        answer    = answer,
        chunks    = chunks,
        reference = tc["reference"]
    )
    all_results.append({
        "question" : tc["question"],
        "reference": tc["reference"],
        "answer"   : answer,
        "metrics"  : metrics
    })
    print(f"  Faithfulness     : {metrics['faithfulness']:.4f}")
    print(f"  Answer Relevancy : {metrics['answer_relevancy']:.4f}")
    print(f"  Context Precision: {metrics['context_precision']:.4f}")
    print(f"  Groundedness     : {metrics['groundedness']:.4f}")
    print()

# Aggregate
print("=" * 50)
print("AGGREGATE SCORES")
print("=" * 50)
for key in ["faithfulness", "answer_relevancy", "context_precision", "groundedness"]:
    scores = [r["metrics"][key] for r in all_results]
    print(f"  {key:<20} : {np.mean(scores):.4f} (min {min(scores):.4f} / max {max(scores):.4f})")

# Save
eval_path = f"{PROJECT_DIR}/evaluation_results_v2.json"
with open(eval_path, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nSaved → {eval_path}")

Re-running evaluation with fixed NLI...

[1/10] What is CEMILAC and what is its primary role?...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Faithfulness     : 0.0000
  Answer Relevancy : 0.9064
  Context Precision: 0.7850
  Groundedness     : 0.1297

AGGREGATE SCORES
  faithfulness         : 0.5126 (min 0.0000 / max 0.9921)
  answer_relevancy     : 0.8496 (min 0.7754 / max 0.9064)
  context_precision    : 0.7383 (min 0.6817 / max 0.7941)
  groundedness         : 0.3172 (min 0.0000 / max 0.7411)

Saved → /kaggle/working/rag_project/evaluation_results_v2.json
